## Initialize Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITR1_amritap1")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "2")
    .config("spark.executor.instances", "2")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/23 04:39:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load Raw Data

In [2]:
commentsDF = spark.read.json("/home/amritap1/30123/final/comments")

In [3]:
submissionsDF = spark.read.json("/home/amritap1/30123/final/submissions")

25/10/23 04:39:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
commentsDF.printSchema()

root
 |-- approved_at_utc: string (nullable = true)
 |-- approved_by: string (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- author_cakeday: boolean (nullable = true)
 |-- author_created_utc: long (nullable = true)
 |-- author_flair_background_color: string (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_richtext: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- e: string (nullable = true)
 |    |    |-- t: string (nullable = true)
 |    |    |-- u: string (nullable = true)
 |-- author_flair_template_id: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- author_flair_text_color: string (nullable = true)
 |-- author_flair_type: string (nullable = true)
 |-- author_fullname: string (nullable = true)
 |-- author_patreon_flair: boolean (nullable = true)
 |-- banned_at_utc: string (nullabl

In [5]:
submissionsDF.printSchema()

root
 |-- approved_at_utc: string (nullable = true)
 |-- approved_by: string (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- author_cakeday: boolean (nullable = true)
 |-- author_created_utc: long (nullable = true)
 |-- author_flair_background_color: string (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_richtext: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- e: string (nullable = true)
 |    |    |-- t: string (nullable = true)
 |    |    |-- u: string (nullable = true)
 |-- author_flair_template_id: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- author_flair_text_color: string (nullable = true)
 |-- author_flair_type: string (nullable = true)
 |-- author_fullname: string (nullable = true)
 |-- author_patreon_flair: boolean (nullable = true)
 |-- banned_at_utc: string (nullabl

## Data Cleaning

I implemented a two-step cleaning process:
1. **Remove bot accounts** - Filter out automoderator, bot, and moderator accounts
2. **Remove deleted content** - Drop `[deleted]` and `[removed]` posts

Keeping bots would artificially inflate certain ticker associations (e.g., automated stock updates), making the patterns less meaningful for understanding organic user discussions.

In [6]:
from pyspark.sql import functions as F

def remove_bot_accounts(df):
    a = F.lower(F.col("author"))
    is_bot = F.coalesce(
        a.contains("automoderator") | a.contains("moderator") | a.contains("bot") | a.contains("automod"),
        F.lit(False)
    )
    df = df.filter(~is_bot)
    return df

In [7]:
commentsDF = remove_bot_accounts(commentsDF)
submissionsDF = remove_bot_accounts(submissionsDF)

In [8]:
from pyspark.sql import functions as F

def remove_deleted_content(df, text_column):
    df = df.filter(
        F.col(text_column).isNotNull() &
        ~F.col(text_column).isin("[deleted]", "[removed]")
    )
    return df

In [9]:
commentsDF = remove_deleted_content(commentsDF, 'body')
submissionsDF = remove_deleted_content(submissionsDF, 'selftext')

## Data Preparation

- **Comments**: Extract `id`, mark source as "comment", use `body` as text
- **Submissions**: Extract `id`, mark source as "submission", concatenate `title` and `selftext`

In [10]:
comments = commentsDF.select(
    F.col("id").alias("id"),
    F.lit("comment").alias("src"),
    F.col("body").alias("text")
).where(F.col("text").isNotNull())

submissions = submissionsDF.select(
    F.col("id").alias("id"),
    F.lit("submission").alias("src"),
    F.concat_ws(" ", F.coalesce("title", F.lit("")), F.coalesce("selftext", F.lit(""))).alias("text")
).where(F.col("text").isNotNull())

In [11]:
comments.printSchema()

root
 |-- id: string (nullable = true)
 |-- src: string (nullable = false)
 |-- text: string (nullable = true)



In [12]:
submissions.printSchema()

root
 |-- id: string (nullable = true)
 |-- src: string (nullable = false)
 |-- text: string (nullable = false)



# Mining Frequent Itemsets

In [13]:
from pyspark.sql import functions as F, types as T

tickers = ["AAPL","TSLA","NVDA","GME","AMC","MSFT"]
ticker_set = set(tickers)

@F.udf(T.ArrayType(T.StringType()))
def extract_simple_tickers(text: str):
    if not text:
        return []
    s = text.upper()
    found = [t for t in ticker_set if t in s]
    return [f"T_{t}" for t in found]

In [14]:
submissions = submissions.select(
    "id",
    extract_simple_tickers(F.col("text")).alias("tickers")
)

In [15]:
comments = comments.select(
    "id",
    extract_simple_tickers("text").alias("tickers")
)

## Creating Buckets

In [16]:
comment_groups = (comments
    .groupBy("id")
    .agg(F.array_distinct(F.flatten(F.collect_list("tickers"))).alias("c_tickers")))

In [17]:
baskets = (submissions.alias("s")
    .join(comment_groups.alias("c"), F.col("s.id") == F.col("c.id"), "left")
    .select(
        F.col("s.id"),
        F.array_distinct(F.array_union(F.coalesce(F.col("s.tickers"), F.array()),
                                       F.coalesce(F.col("c.c_tickers"), F.array()))).alias("items"))
    .filter(F.size("items") >= 2))

## FP-Growth

### Limited Ticker Set

In [18]:
from pyspark.ml.fpm import FPGrowth

fp = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.1).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))

In [19]:
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))

In [20]:
pairs.count()

15

In [21]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
|[T_AMC, T_GME]  |1   |
|[T_AMC, T_TSLA] |1   |
+----------------+----+



In [22]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+--------------------------------+----------+----------+------------------+---------------------+
|antecedent                      |consequent|confidence|lift              |support              |
+--------------------------------+----------+----------+------------------+---------------------+
|[T_MSFT, T_NVDA, T_TSLA]        |[T_AMC]   |0.25      |8.59375           |0.0036363636363636364|
|[T_MSFT, T_NVDA, T_TSLA, T_AAPL]|[T_AMC]   |0.25      |8.59375           |0.0036363636363636364|
|[T_AMC, T_TSLA, T_AAPL]         |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_AAPL]         |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA, T_AAPL] |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA]         |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_TSLA]                 |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_TSLA, T_A

**Initial parameters**:
- `minSupport = 0.001` (0.1%) - Very low threshold to capture even rare associations
- `minConfidence = 0.1` (10%) - Low confidence to see all potential patterns

Used a small set of popular tickers: `["AAPL","TSLA","NVDA","GME","AMC","MSFT"]`

**Outcome**: The model identified frequent pairs and generated association rules.

**Challenge**: Simple substring matching could lead to false positives (e.g., "GME" might match "SEGMENT")

In [23]:
fp = FPGrowth(itemsCol="items", minSupport=0.002, minConfidence=0.3).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
pairs.count()

15

In [24]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
|[T_AMC, T_GME]  |1   |
|[T_AMC, T_TSLA] |1   |
+----------------+----+



In [25]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+-------------------------------+----------+----------+------------------+---------------------+
|antecedent                     |consequent|confidence|lift              |support              |
+-------------------------------+----------+----------+------------------+---------------------+
|[T_AMC, T_TSLA, T_AAPL]        |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA]        |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_AAPL]        |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA, T_AAPL]|[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_TSLA]                |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_MSFT, T_TSLA]        |[T_NVDA]  |1.0       |2.1484375         |0.0036363636363636364|
|[T_AMC, T_MSFT, T_TSLA, T_AAPL]|[T_NVDA]  |1.0       |2.1484375         |0.0036363636363636364|
|[T_AMC, T_TSLA]              

**Adjustment**: Increased thresholds to filter out noise
- `minSupport = 0.002` (0.2%) - Doubled the support threshold
- `minConfidence = 0.3` (30%) - Tripled confidence to focus on stronger associations

**Test**: Whether tightening parameters reveals clearer market sentiment patterns (e.g., stocks in the same sector being discussed together).

**Outcome**: Result is largely similar

In [26]:
fp = FPGrowth(itemsCol="items", minSupport=0.005, minConfidence=0.2).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
pairs.count()

13

In [27]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
+----------------+----+



In [28]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+------------------------+----------+-------------------+------------------+--------------------+
|antecedent              |consequent|confidence         |lift              |support             |
+------------------------+----------+-------------------+------------------+--------------------+
|[T_MSFT, T_NVDA, T_TSLA]|[T_AAPL]  |1.0                |1.4473684210526316|0.014545454545454545|
|[T_AMC, T_AAPL]         |[T_MSFT]  |0.5                |1.3349514563106795|0.007272727272727273|
|[T_GME, T_TSLA, T_AAPL] |[T_MSFT]  |0.5                |1.3349514563106795|0.007272727272727273|
|[T_GME, T_AAPL]         |[T_MSFT]  |0.45454545454545453|1.2135922330097086|0.01818181818181818 |
|[T_GME]                 |[T_MSFT]  |0.4375             |1.1680825242718447|0.05090909090909091 |
|[T_MSFT]                |[T_AAPL]  |0.7378640776699029 |1.0679611650485437|0.27636363636363637 |
|[T_AAPL]                |[T_MSFT]  |0.4                |1.0679611650485437|0.27636363636363637 |
|[T_AMC]            

**Further adjustment**: 
- `minSupport = 0.005` (0.5%) - Even higher support threshold
- `minConfidence = 0.2` (20%) - Slightly lowered confidence from Iteration 2

**Test**: A higher support threshold ensures we only look at frequently co-occurring tickers, while allowing slightly lower confidence captures diverse association patterns.

**Outcome**: Results slightly narrowed down but largely similar

In [29]:
fp = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.05).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
pairs.count()

13

In [30]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
+----------------+----+



In [31]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+------------------------+----------+-------------------+------------------+--------------------+
|antecedent              |consequent|confidence         |lift              |support             |
+------------------------+----------+-------------------+------------------+--------------------+
|[T_MSFT, T_NVDA, T_TSLA]|[T_AAPL]  |1.0                |1.4473684210526316|0.014545454545454545|
|[T_MSFT, T_TSLA]        |[T_GME]   |0.14285714285714285|1.2276785714285714|0.014545454545454545|
|[T_GME, T_AAPL]         |[T_MSFT]  |0.45454545454545453|1.2135922330097086|0.01818181818181818 |
|[T_MSFT]                |[T_GME]   |0.13592233009708737|1.1680825242718447|0.05090909090909091 |
|[T_GME]                 |[T_MSFT]  |0.4375             |1.1680825242718447|0.05090909090909091 |
|[T_MSFT]                |[T_AAPL]  |0.7378640776699029 |1.0679611650485437|0.27636363636363637 |
|[T_AAPL]                |[T_MSFT]  |0.4                |1.0679611650485437|0.27636363636363637 |
|[T_AMC]            

In [32]:
rules.orderBy("support").select("antecedent", "consequent", "support", "confidence").show(10)

+--------------------+----------+--------------------+-------------------+
|          antecedent|consequent|             support|         confidence|
+--------------------+----------+--------------------+-------------------+
|             [T_AMC]|  [T_NVDA]| 0.01090909090909091|              0.375|
|             [T_AMC]|  [T_MSFT]| 0.01090909090909091|              0.375|
|[T_MSFT, T_NVDA, ...|  [T_TSLA]|0.014545454545454545|0.36363636363636365|
|     [T_GME, T_AAPL]|  [T_TSLA]|0.014545454545454545|0.36363636363636365|
|     [T_GME, T_TSLA]|  [T_AAPL]|0.014545454545454545| 0.3333333333333333|
|    [T_MSFT, T_TSLA]|   [T_GME]|0.014545454545454545|0.14285714285714285|
|     [T_GME, T_TSLA]|  [T_MSFT]|0.014545454545454545| 0.3333333333333333|
|[T_MSFT, T_NVDA, ...|  [T_AAPL]|0.014545454545454545|                1.0|
|             [T_AMC]|  [T_AAPL]|0.014545454545454545|                0.5|
|    [T_MSFT, T_TSLA]|  [T_NVDA]|0.014545454545454545|0.14285714285714285|
+--------------------+---

In [33]:
rules.orderBy("confidence").select("antecedent", "consequent", "support", "confidence").show(10)

+----------------+----------+--------------------+-------------------+
|      antecedent|consequent|             support|         confidence|
+----------------+----------+--------------------+-------------------+
|        [T_NVDA]|   [T_GME]|0.025454545454545455|          0.0546875|
|        [T_AAPL]|   [T_GME]|                0.04|0.05789473684210526|
|[T_NVDA, T_TSLA]|  [T_MSFT]|0.014545454545454545|0.06557377049180328|
|[T_MSFT, T_AAPL]|   [T_GME]| 0.01818181818181818|0.06578947368421052|
|        [T_TSLA]|   [T_GME]| 0.04363636363636364|0.08333333333333333|
|        [T_MSFT]|   [T_GME]| 0.05090909090909091|0.13592233009708737|
|[T_MSFT, T_TSLA]|   [T_GME]|0.014545454545454545|0.14285714285714285|
|[T_MSFT, T_TSLA]|  [T_NVDA]|0.014545454545454545|0.14285714285714285|
|[T_MSFT, T_AAPL]|  [T_NVDA]|                0.04|0.14473684210526316|
|[T_NVDA, T_AAPL]|  [T_MSFT]|                0.04|0.16923076923076924|
+----------------+----------+--------------------+-------------------+
only s

**Parameter choice**:
- `minSupport = 0.01` (1%) - Highest threshold yet, focusing on only the most common patterns
- `minConfidence = 0.05` (5%) - Very low confidence to see all patterns that meet support

**Test**: If extremely high support with low confidence reveals the "core" ticker associations - patterns that appear frequently enough to be statistically significant, regardless of their predictive power. Also looking for 3+ ticker combinations to find more complex discussion patterns

**Outcome**: Results largely similar to the previous case

In [34]:
fp = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.05).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") >= 3)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
print(pairs.count())
pairs.show(20, truncate=False)

[Stage 156:>                                                        (0 + 4) / 4]

8
+--------------------------------+----+
|pair                            |freq|
+--------------------------------+----+
|[T_AAPL, T_MSFT, T_TSLA]        |19  |
|[T_AAPL, T_NVDA, T_TSLA]        |16  |
|[T_AAPL, T_MSFT, T_NVDA]        |11  |
|[T_AAPL, T_GME, T_MSFT]         |5   |
|[T_MSFT, T_NVDA, T_TSLA]        |4   |
|[T_AAPL, T_GME, T_TSLA]         |4   |
|[T_AAPL, T_MSFT, T_NVDA, T_TSLA]|4   |
|[T_GME, T_MSFT, T_TSLA]         |4   |
+--------------------------------+----+



**Test**: Looking at triplets (3+ tickers) to find complex co-mention patterns.

**Outcome**: The output likely shows very few or no triplets because:
1. With only 6 tickers, we have limited combinations (20 possible triplets)
2. The high support threshold (1%) means triplets need to appear in many submissions

### Expanded Ticker Set

In [36]:
comments = commentsDF.select(
    F.col("id").alias("id"),
    F.lit("comment").alias("src"),
    F.col("body").alias("text")
).where(F.col("text").isNotNull())

submissions = submissionsDF.select(
    F.col("id").alias("id"),
    F.lit("submission").alias("src"),
    F.concat_ws(" ", F.coalesce("title", F.lit("")), F.coalesce("selftext", F.lit(""))).alias("text")
).where(F.col("text").isNotNull())

In [37]:
from pyspark.sql import functions as F, types as T

tickers = ["AAPL", "AMZN", "MSFT", "GOOG", "META", "TXN", "ADP", "BSX", "APH", "ISRG", "GILD", "DE", "SYK", "ETN", "COF", "LLY", "UNH", "LOW", "HON", "PG", "ADI", "SCHW", "PLD", "CTRA", "KKR"]
ticker_set = set(tickers)

@F.udf(T.ArrayType(T.StringType()))
def extract_simple_tickers(text: str):
    if not text:
        return []
    s = text.upper()
    found = [t for t in ticker_set if t in s]
    return [f"T_{t}" for t in found]

In [38]:
submissions = submissions.select(
    "id",
    extract_simple_tickers(F.col("text")).alias("tickers")
)

In [39]:
comments = comments.select(
    "id",
    extract_simple_tickers("text").alias("tickers")
)

In [40]:
comment_groups = (comments
    .groupBy("id")
    .agg(F.array_distinct(F.flatten(F.collect_list("tickers"))).alias("c_tickers")))

In [41]:
baskets = (submissions.alias("s")
    .join(comment_groups.alias("c"), F.col("s.id") == F.col("c.id"), "left")
    .select(
        F.col("s.id"),
        F.array_distinct(F.array_union(F.coalesce(F.col("s.tickers"), F.array()),
                                       F.coalesce(F.col("c.c_tickers"), F.array()))).alias("items"))
    .filter(F.size("items") >= 2))

In [42]:
fp = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.1).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))

In [43]:
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))

In [44]:
pairs.show(20, truncate=False)

[Stage 189:>                                                        (0 + 4) / 4]

+---------------+----+
|pair           |freq|
+---------------+----+
|[T_DE, T_LLY]  |7933|
|[T_DE, T_LOW]  |5974|
|[T_ADI, T_DE]  |4967|
|[T_LLY, T_LOW] |3243|
|[T_ADI, T_LLY] |2494|
|[T_ADI, T_LOW] |2039|
|[T_DE, T_HON]  |1157|
|[T_DE, T_GOOG] |949 |
|[T_DE, T_PG]   |862 |
|[T_AAPL, T_DE] |722 |
|[T_HON, T_LLY] |633 |
|[T_AMZN, T_DE] |521 |
|[T_GOOG, T_LLY]|489 |
|[T_HON, T_LOW] |489 |
|[T_APH, T_DE]  |472 |
|[T_ADI, T_HON] |370 |
|[T_GOOG, T_LOW]|361 |
|[T_LLY, T_PG]  |338 |
|[T_AAPL, T_LLY]|331 |
|[T_LOW, T_PG]  |314 |
+---------------+----+
only showing top 20 rows



In [45]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+------------------------------------+----------+-------------------+------------------+---------------------+
|antecedent                          |consequent|confidence         |lift              |support              |
+------------------------------------+----------+-------------------+------------------+---------------------+
|[T_AAPL, T_ADI, T_LOW, T_DE]        |[T_COF]   |0.17               |20.607642857142856|0.0010017087973602026|
|[T_AMZN, T_AAPL, T_GOOG]            |[T_MSFT]  |0.375              |20.332667731629392|0.001590949266395616 |
|[T_AMZN, T_AAPL, T_LOW, T_DE]       |[T_MSFT]  |0.3559322033898305 |19.298803270699086|0.001237404984974368 |
|[T_AAPL, T_ADI, T_LOW]              |[T_COF]   |0.1574074074074074 |19.081150793650792|0.0010017087973602026|
|[T_AMZN, T_AAPL, T_LOW]             |[T_MSFT]  |0.34375            |18.63827875399361 |0.0012963290318779093|
|[T_AMZN, T_AAPL, T_LLY]             |[T_MSFT]  |0.34285714285714286|18.589867640346874|0.0014141771256849921|
|

In [46]:
rules.orderBy("support").select("antecedent", "consequent", "support", "confidence").show(10)

+--------------------+----------+--------------------+-------------------+
|          antecedent|consequent|             support|         confidence|
+--------------------+----------+--------------------+-------------------+
|[T_COF, T_AAPL, T...|   [T_LOW]|0.001001708797360...| 0.9444444444444444|
|[T_GOOG, T_HON, T...|    [T_PG]|0.001001708797360...| 0.4146341463414634|
|[T_AMZN, T_GOOG, ...|  [T_AAPL]|0.001001708797360...| 0.7083333333333334|
|[T_AAPL, T_GOOG, ...|   [T_ADI]|0.001001708797360...| 0.5666666666666667|
|[T_AAPL, T_HON, T...|  [T_GOOG]|0.001001708797360...| 0.3617021276595745|
|[T_PG, T_AAPL, T_...|   [T_LOW]|0.001001708797360...| 0.9444444444444444|
|[T_APH, T_PG, T_HON]|    [T_DE]|0.001001708797360...|                1.0|
|    [T_MSFT, T_GOOG]|   [T_HON]|0.001001708797360...|0.23943661971830985|
|     [T_GILD, T_LOW]|   [T_ADI]|0.001001708797360...| 0.3469387755102041|
|[T_PG, T_AAPL, T_...|   [T_LLY]|0.001001708797360...| 0.9444444444444444|
+--------------------+---

In [47]:
rules.orderBy("confidence").select("antecedent", "consequent", "support", "confidence").show(10)

+--------------------+----------+--------------------+-------------------+
|          antecedent|consequent|             support|         confidence|
+--------------------+----------+--------------------+-------------------+
|[T_GOOG, T_LOW, T...|  [T_MSFT]|0.001355253078781...|                0.1|
|      [T_HON, T_LOW]|   [T_APH]|0.002887278298273...|0.10020449897750511|
| [T_PG, T_LOW, T_DE]|  [T_AMZN]|0.001708797360202...|0.10034602076124567|
|[T_HON, T_ADI, T_...|   [T_COF]|0.001237404984974368|0.10144927536231885|
|      [T_ADI, T_LOW]|   [T_HON]|0.012197277709033056| 0.1015203531142717|
|[T_ADI, T_LOW, T_...|  [T_GOOG]|0.006776265393907253|0.10159010600706714|
|       [T_PG, T_LOW]|   [T_APH]|0.001885569500913...|0.10191082802547771|
|        [T_PG, T_DE]|   [T_HON]|0.005185316127511637|0.10208816705336426|
|      [T_HON, T_LLY]|    [T_PG]|0.003830063048730187|0.10268562401263823|
|       [T_HON, T_DE]|  [T_GOOG]|0.007011961581521419|0.10285220397579949|
+--------------------+---

In [48]:
pairs = (freq
    .filter(F.size("items") >= 3)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
print(pairs.count())
pairs.show(20, truncate=False)

576
+---------------------------+----+
|pair                       |freq|
+---------------------------+----+
|[T_DE, T_LLY, T_LOW]       |2798|
|[T_ADI, T_DE, T_LLY]       |2172|
|[T_ADI, T_DE, T_LOW]       |1776|
|[T_ADI, T_LLY, T_LOW]      |1192|
|[T_ADI, T_DE, T_LLY, T_LOW]|1132|
|[T_DE, T_HON, T_LLY]       |560 |
|[T_DE, T_HON, T_LOW]       |438 |
|[T_DE, T_GOOG, T_LLY]      |429 |
|[T_ADI, T_DE, T_HON]       |329 |
|[T_DE, T_GOOG, T_LOW]      |323 |
|[T_DE, T_LLY, T_PG]        |309 |
|[T_HON, T_LLY, T_LOW]      |307 |
|[T_DE, T_HON, T_LLY, T_LOW]|296 |
|[T_DE, T_LOW, T_PG]        |289 |
|[T_ADI, T_DE, T_GOOG]      |263 |
|[T_AAPL, T_DE, T_LLY]      |250 |
|[T_APH, T_DE, T_LLY]       |250 |
|[T_ADI, T_HON, T_LLY]      |245 |
|[T_ADI, T_DE, T_PG]        |235 |
|[T_ADI, T_DE, T_HON, T_LLY]|231 |
+---------------------------+----+
only showing top 20 rows



**Ticker frequencies**: If a stock is mentioned very frequently, it will naturally appear in many pairs - but that doesn't necessarily mean meaningful association.

In [49]:
tick_freq = (baskets
    .select(F.explode("items").alias("ticker"))
    .groupBy("ticker")
    .count()
    .orderBy(F.desc("count"))
)
tick_freq.show(15, truncate=False)

[Stage 226:==============>                                          (1 + 3) / 4]

+------+-----+
|ticker|count|
+------+-----+
|T_DE  |15245|
|T_LLY |8889 |
|T_LOW |6816 |
|T_ADI |5637 |
|T_HON |1365 |
|T_GOOG|1149 |
|T_AAPL|991  |
|T_PG  |952  |
|T_AMZN|735  |
|T_APH |540  |
|T_MSFT|313  |
|T_META|211  |
|T_COF |140  |
|T_GILD|125  |
|T_ETN |115  |
+------+-----+
only showing top 15 rows



**Pair Frequencies**: Building all possible ticker pairs manually using `combinations` and counting their co-occurrence frequency. The raw co-occurrence counts without FP-Growth's support/confidence filtering, helping us understand if the algorithm is appropriately filtering or if we're missing important patterns.

In [50]:
from itertools import combinations
from pyspark.sql.types import ArrayType, StringType

pairs_udf = F.udf(lambda xs: [tuple(sorted(x)) for x in combinations(set(xs), 2)],
                  ArrayType(ArrayType(StringType())))

pair_freq = (baskets
    .select(F.explode(pairs_udf("items")).alias("pair"))
    .select(F.col("pair")[0].alias("t1"), F.col("pair")[1].alias("t2"))
    .groupBy("t1","t2").count()
    .orderBy(F.desc("count"))
)
pair_freq.show(15, truncate=False)

[Stage 235:>                                                        (0 + 4) / 4]

+------+------+-----+
|t1    |t2    |count|
+------+------+-----+
|T_DE  |T_LLY |7933 |
|T_DE  |T_LOW |5974 |
|T_ADI |T_DE  |4967 |
|T_LLY |T_LOW |3243 |
|T_ADI |T_LLY |2494 |
|T_ADI |T_LOW |2039 |
|T_DE  |T_HON |1157 |
|T_DE  |T_GOOG|949  |
|T_DE  |T_PG  |862  |
|T_AAPL|T_DE  |722  |
|T_HON |T_LLY |633  |
|T_AMZN|T_DE  |521  |
|T_HON |T_LOW |489  |
|T_GOOG|T_LLY |489  |
|T_APH |T_DE  |472  |
+------+------+-----+
only showing top 15 rows



## Correlation Analysis

Are the association rules just reflecting ticker popularity, or are they revealing genuine relationships?

**High correlation** = Rules mostly reflect popular stocks appearing together by chance

**Low correlation** = Rules capture genuine co-mention patterns beyond base popularity

If correlation is high, we need to normalize or use lift more heavily to find meaningful associations.

In [51]:
lhs_freq = tick_freq.select(F.col("ticker").alias("lhs"), F.col("count").alias("lhs_freq"))

rules_pop = (rules
    .where(F.size("antecedent")==1)
    .select(F.col("antecedent")[0].alias("lhs"), "support","confidence")
    .join(lhs_freq, "lhs", "left")
)

rules_pop.agg(
    F.corr("lhs_freq","support").alias("corr_support"),
    F.corr("lhs_freq","confidence").alias("corr_confidence")
).show()

+------------------+--------------------+
|      corr_support|     corr_confidence|
+------------------+--------------------+
|0.9111743111495563|0.042835502183103474|
+------------------+--------------------+



**Outcome**:

Very strong positive correlation - Rules involving more popular tickers also tend to have higher support — i.e., those tickers appear in a larger fraction of baskets, so their rules are more frequent. Popularity directly drives support.

Near-zero correlation - Popularity of a ticker has almost no effect on rule confidence — common tickers don’t necessarily make more reliable or predictive rules; they just occur more often.

## Experimentation Summary

Testing multiple combinations of `minSupport` and `minConfidence` helped understand the trade-off between pattern abundance and pattern significance.

Moving from 6 to 25 tickers dramatically increased the richness of the analysis, revealing more nuanced market discussion patterns.

Building pairs manually and calculating correlations helped verify that FP-Growth wasn't just surfacing artifacts of popularity.

Looking for 3+ ticker combinations yielded sparse results which could be due to the nature of discussions available in the data - Reddit discussions typically focus on comparing 2 stocks or deep-diving on 1

Substring matching for tickers risks false positives (e.g., "DE" matching "DELETED"). A more robust approach would use word boundaries or NLP-based entity extraction.